### Contextual Embedding Extraction

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
from pathlib import Path
import torch
from tqdm import tqdm
import pickle
import json
from collections import defaultdict

model_dir = Path("./.cache/models")

model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "Qwen/Qwen3-8B"
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, device_map="auto", trust_remote_code=True, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir, trust_remote_code=True)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
if "Llama" in model_name:
    system_prompt = f"""You are a logical reasoning assistant. You are given a logic puzzle.
Guidelines:
- All constraints come **only** from the <conditions> section.
- The <scenario> provides background and intuition, but **does not impose any additional rules or constraints**.
- All variables represent **independent decisions**; there is no mutual exclusivity or implicit linkage unless stated explicitly in <conditions>.
- Variables not mentioned in <conditions> are considered unknown and irrelevant to satisfiability.
Your task:
- If the puzzle is satisfiable, propose one valid assignment that satisfies all the conditions.
- If the puzzle is unsatisfiable, explain why some of the conditions cannot all be true at once.
Think step by step but don't return any intermediate reasoning, only give a brief response. At the end of your response, output exactly one of the following labels on a new line:
[SAT]- if a valid assignment exists
[UNSAT]- if the constraints cannot be satisfied
Do not add any text or formatting after the final label.
"""
elif "Qwen" in model_name:
    system_prompt = f"""You are a logical reasoning assistant. You are given a logic puzzle.
Guidelines:
- All constraints come **only** from the <conditions> section.
- The <scenario> provides background and intuition, but **does not impose any additional rules or constraints**.
- All variables represent **independent decisions**; there is no mutual exclusivity or implicit linkage unless stated explicitly in <conditions>.
- Variables not mentioned in <conditions> are considered unknown and irrelevant to satisfiability.
Your task:
- If the puzzle is satisfiable, propose one valid assignment that satisfies all the conditions.
- If the puzzle is unsatisfiable, explain why some of the conditions cannot all be true at once.
Think step by step but don't return any intermediate reasoning, summarize the key points, and only give a brief response.
Do not use conversational filler, introductions, or transitional phrasing like 'Let's look at this' or 'Let's analyze the conditions'. Start immediately with the core facts.
At the end of your response, output exactly one of the following labels on a new line:
[SAT] - if a valid assignment exists
[UNSAT] - if the constraints cannot be satisfied
Do not add any text or formatting after the final label.
"""
else:
    system_prompt = f"""You are a logical reasoning assistant. You are given a logic puzzle.
Guidelines:
- All constraints come **only** from the <conditions> section.
- The <scenario> provides background and intuition, but **does not impose any additional rules or constraints**.
- All variables represent **independent decisions**; there is no mutual exclusivity or implicit linkage unless stated explicitly in <conditions>.
- Variables not mentioned in <conditions> are considered unknown and irrelevant to satisfiability.
Your task:
- If the puzzle is satisfiable, propose one valid assignment that satisfies all the conditions.
- If the puzzle is unsatisfiable, explain why some of the conditions cannot all be true at once.
Think step by step but don't return any intermediate reasoning, summarize the key points, and only give a brief response. At the end of your response, output exactly one of the following labels on a new line:
[SAT] - if a valid assignment exists
[UNSAT] - if the constraints cannot be satisfied
Do not add any text or formatting after the final label.
"""

Hidden states of all generated tokens

In [3]:
system_prompt = f"""You are a logical reasoning assistant. You are given a logic puzzle.
Guidelines:
- All constraints come **only** from the <conditions> section.
- The <scenario> provides background and intuition, but **does not impose any additional rules or constraints**.
- All variables represent **independent decisions**; there is no mutual exclusivity or implicit linkage unless stated explicitly in <conditions>.
- Variables not mentioned in <conditions> are considered unknown and irrelevant to satisfiability.
Your task: Think step by step but don't return any intermediate reasoning, and only give a brief response following these rules:
- If the puzzle is satisfiable, response only the final label [SAT].
- If the puzzle is unsatisfiable, response only the final label [UNSAT].
Do not add any text or formatting after the final label.
"""

In [7]:
import re

results = defaultdict(list)
data = json.load(open("./results/2100_samples_SATBench_Llama3.1-8B/logical_result_max128_temp0.0_verdict_only.json", "r"))
index = 0
system_prompt = system_prompt

for item in tqdm(data):
    if item["pred_label"]:
        scenario = item["scenario"]
        conditions = item["conditions"]
        question = item["question"]
        user_prompt = f"""<scenario>
{scenario}
<conditions>
{conditions}
<question>
{question}
"""
        trace = item["model_trace"]
        correct = item["correct_prediction"]
        results['correct'].append(correct)

        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]

        full_messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": trace
            }
        ]

        tokenized_query = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).to(model.device)
        prompt_length = tokenized_query.input_ids.shape[-1]
        full_inputs = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).input_ids.to(model.device)
        kwargs = {
            "input_ids": full_inputs,
            "use_cache": False,
            "past_key_values": None,
            "output_hidden_states": True,
            "return_dict": True,
        }
        with torch.no_grad():
            outputs = model(**kwargs)
            generated_embeddings = outputs.hidden_states[-1][0][prompt_length-1:-1]
            generated_embeddings = generated_embeddings.to(torch.float32).detach().cpu().numpy()

        # hidden_size = outputs.hidden_states[-1].shape[-1]
        # generated_embeddings = np.stack(step_embeddings) if step_embeddings else np.empty((0, hidden_size), dtype=np.float32)
        # print(f"Sample {index}: Generated embeddings shape: {generated_embeddings.shape}")
        # print("*" * 50)
        results['generated_embeddings'].append(generated_embeddings)
        results['index'].append(index)
    index += 1
with open("./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max128_temp0.0_verdict_only.pkl", "wb") as outfile:
    outfile.write(pickle.dumps(results))

100%|██████████| 2100/2100 [14:22<00:00,  2.43it/s]


In [1]:
ls = [1,2,3,6,8,9]
print(ls[3:-1])

[6, 8]


Hidden states of each step in CoT

In [10]:
import re

results = defaultdict(list)
data = json.load(open("./results/2100_samples_SATBench_Llama3.1-8B/logical_result_max512_temp0.0_v1.json", "r"))
index = 0
system_prompt = system_prompt


def split_trace_into_spans(trace):
    verdict_match = re.search(r"\s*\[(SAT|UNSAT)\]\s*$", trace)
    reasoning = trace[:verdict_match.start()] if verdict_match else trace
    lines = [(match.group().strip(), match.start(), match.end())
             for match in re.finditer(r"[^\n]+", reasoning)
             if match.group().strip()]
    if len(lines) > 1:
        return reasoning, lines

    sentences = [(match.group().strip(), match.start(), match.end())
                 for match in re.finditer(r".+?(?:[.!?](?=\s|$)|$)", reasoning.strip())
                 if match.group().strip()]
    return reasoning, sentences


for item in tqdm(data):
    if item["pred_label"] and len(item["model_trace"].strip()) > 7:
        scenario = item["scenario"]
        conditions = item["conditions"]
        question = item["question"]
        user_prompt = f"""<scenario>
{scenario}
<conditions>
{conditions}
<question>
{question}
"""
        trace = item["model_trace"]
        correct = item["correct_prediction"]
        results['correct'].append(correct)

        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]

        full_messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": trace
            }
        ]

        tokenized_query = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).to(model.device)
        prompt_length = tokenized_query.input_ids.shape[-1]
        full_inputs = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).input_ids.to(model.device)
        kwargs = {
            "input_ids": full_inputs,
            "use_cache": False,
            "past_key_values": None,
            "output_hidden_states": True,
            "return_dict": True,
        }
        with torch.no_grad():
            outputs = model(**kwargs)

        reasoning, step_spans = split_trace_into_spans(trace)
        # print(f"Processing sample {index}: {len(step_spans)} steps found in the reasoning trace.")
        # print(f"Reasoning trace:\n{reasoning}\n")
        # print(f"Step spans: {step_spans}\n")
        trace_start = prompt_length - 1
        previous_token_end = 0
        step_embeddings = []
        for _, _, step_end_char in step_spans:
            cumulative_text = reasoning[:step_end_char]
            cumulative_token_end = len(tokenizer(
                cumulative_text,
                add_special_tokens=False
            ).input_ids)
            step_start = trace_start + previous_token_end
            step_stop = trace_start + cumulative_token_end
            if step_stop > step_start:
                # step_hidden_states = outputs.hidden_states[-1][0][step_start:step_stop]
                # pooled_step = step_hidden_states.to(torch.float32).mean(dim=0).detach().cpu().numpy()
                delimiter_idx = step_stop - 1  # Extract the state of the final token in the step (delimiter)
                step_hidden_states = outputs.hidden_states[-12][0][delimiter_idx]
                pooled_step = step_hidden_states.to(torch.float32).detach().cpu().numpy()
                step_embeddings.append(pooled_step)
            previous_token_end = cumulative_token_end

        hidden_size = outputs.hidden_states[-1].shape[-1]
        generated_embeddings = np.stack(step_embeddings) if step_embeddings else np.empty((0, hidden_size), dtype=np.float32)
        # print(f"Sample {index}: Generated embeddings shape: {generated_embeddings.shape}")
        results['generated_embeddings'].append(generated_embeddings)
        results['index'].append(index)
    index += 1
with open("./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max512_temp0.0_delimiter_steps_layer21.pkl", "wb") as outfile:
    outfile.write(pickle.dumps(results))

100%|██████████| 2100/2100 [12:51<00:00,  2.72it/s]


### Extract LLM output token confidence

In [31]:
import re
import torch.nn.functional as F

results = defaultdict(list)
data = json.load(open("./results/2100_samples_SATBench_Llama3.1-8B/logical_result_max512_temp0.0_v1.json", "r"))
index = 0
system_prompt = system_prompt

sat_token_id = tokenizer.encode("SAT", add_special_tokens=False)[0]
unsat_token_id = tokenizer.encode("UNSAT", add_special_tokens=False)[0]
print(f"SAT token ID: {sat_token_id}, UNSAT token ID: {unsat_token_id}")

for item in tqdm(data):
    if item["pred_label"]:
        scenario = item["scenario"]
        conditions = item["conditions"]
        question = item["question"]
        user_prompt = f"""<scenario>
{scenario}
<conditions>
{conditions}
<question>
{question}
"""
        trace = item["model_trace"]
        correct = item["correct_prediction"]
        if item["gold_label"] == "SAT":
            gold_label = 1
        else:
            gold_label = 0
        if item["pred_label"] == "SAT":
            pred_label = 1
        else:
            pred_label = 0
        results['correct'].append(correct)
        results['ground_truth'].append(gold_label)
        results['llm_prediction'].append(pred_label)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        full_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": trace}
        ]

        tokenized_query = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).to(model.device)
        prompt_length = tokenized_query.input_ids.shape[-1]
        
        full_inputs = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).input_ids.to(model.device)

        kwargs = {
            "input_ids": full_inputs,
            "use_cache": False,
            "past_key_values": None,
            "output_hidden_states": False,
            "return_dict": True,
        }
        with torch.no_grad():
            outputs = model(**kwargs)

        input_ids_list = full_inputs[0].tolist()
        # Look for the '[' character
        bracket_token_id = tokenizer.encode("[", add_special_tokens=False)[0]


        # Restrict the search to the last 10 tokens to avoid intermediate reasoning brackets
        # search_limit = max(prompt_length, len(input_ids_list) - 9)
        verdict_start_idx = -1
        for i in range(len(input_ids_list) - 1, prompt_length - 1, -1):
            token_str = tokenizer.decode([input_ids_list[i]])
            if "[" in token_str:
                verdict_start_idx = i
                break
        # print(f"Sample {index}: Verdict start index: {verdict_start_idx}, Input IDs length: {len(input_ids_list)}") 
        if verdict_start_idx != -1 and verdict_start_idx < len(input_ids_list) - 1:
            # The verdict sequence is everything from '[' to the end of the trace
            verdict_indices = range(verdict_start_idx, len(input_ids_list))
            
            step_probabilities = []
            
            # 3. Calculate the probability the model assigned to each generated token
            for idx in verdict_indices:
                # The logits for the token at `idx` are produced by the model at `idx - 1`
                logits_for_step = outputs.logits[0, idx - 1] 
                
                # Convert logits to a probability distribution over the whole vocabulary
                probs = F.softmax(logits_for_step, dim=-1)

                predicted_token_id = torch.argmax(logits_for_step).item()
                # print(f"Sample {index}: Token index {idx}, Predicted token ID: {predicted_token_id}, Actual token ID: {input_ids_list[idx]}, Probability distribution shape: {probs.shape}")
                step_confidence = torch.max(probs).item()
                step_probabilities.append(step_confidence)
                
                # What token did the model actually generate?
                # actual_token_id = input_ids_list[idx]
                
                # # What was the probability assigned to that specific token?
                # token_prob = probs[actual_token_id].item()
                # step_probabilities.append(token_prob)
                
            # 4. Aggregate the confidence
            # Use the Geometric Mean to prevent long token sequences from penalizing the score
            if len(step_probabilities) > 0:
                product = np.prod(step_probabilities)
                llm_confidence = product ** (1.0 / len(step_probabilities))
            else:
                llm_confidence = 0.0
        else:
            # Fallback if no formatted verdict was found
            llm_confidence = 0.0

        # print(f"Sample {index}: LLM confidence in its answer: {llm_confidence:.4f}")
        # Store this value in your results dictionary
        results['llm_confidence'].append(llm_confidence)
    index += 1
with open("./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max512_temp0.0_logits.pkl", "wb") as outfile:
    outfile.write(pickle.dumps(results))

SAT token ID: 85333, UNSAT token ID: 1899


100%|██████████| 2100/2100 [12:53<00:00,  2.71it/s]


collect the confidence of the whole sequence

In [4]:
import re
import torch.nn.functional as F

results = defaultdict(list)
data = json.load(open("./results/2100_samples_SATBench_Llama3.1-8B/logical_result_max512_temp0.0_v1.json", "r"))
index = 0
system_prompt = system_prompt

sat_token_id = tokenizer.encode("SAT", add_special_tokens=False)[0]
unsat_token_id = tokenizer.encode("UNSAT", add_special_tokens=False)[0]
# print(f"SAT token ID: {sat_token_id}, UNSAT token ID: {unsat_token_id}")

for item in tqdm(data):
    if item["pred_label"]:
        scenario = item["scenario"]
        conditions = item["conditions"]
        question = item["question"]
        user_prompt = f"""<scenario>
{scenario}
<conditions>
{conditions}
<question>
{question}
"""
        trace = item["model_trace"]
        correct = item["correct_prediction"]
        if item["gold_label"] == "SAT":
            gold_label = 1
        else:
            gold_label = 0
        if item["pred_label"] == "SAT":
            pred_label = 1
        else:
            pred_label = 0
        results['correct'].append(correct)
        results['ground_truth'].append(gold_label)
        results['llm_prediction'].append(pred_label)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        full_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": trace}
        ]

        tokenized_query = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).to(model.device)
        prompt_length = tokenized_query.input_ids.shape[-1]
        
        full_inputs = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False
        ).input_ids.to(model.device)

        kwargs = {
            "input_ids": full_inputs,
            "use_cache": False,
            "past_key_values": None,
            "output_hidden_states": False,
            "return_dict": True,
        }
        with torch.no_grad():
            outputs = model(**kwargs)

        input_ids_list = full_inputs[0].tolist()
        # Look for the '[' character
        # bracket_token_id = tokenizer.encode("[", add_special_tokens=False)[0]


        # Restrict the search to the last 10 tokens to avoid intermediate reasoning brackets
        # search_limit = max(prompt_length, len(input_ids_list) - 9)
        verdict_start_idx = prompt_length - 1
        # for i in range(len(input_ids_list) - 1, prompt_length - 1, -1):
        #     token_str = tokenizer.decode([input_ids_list[i]])
        #     if "[" in token_str:
        #         verdict_start_idx = i
        #         break
        # print(f"Sample {index}: Verdict start index: {verdict_start_idx}, Input IDs length: {len(input_ids_list)}") 
        if verdict_start_idx != -1 and verdict_start_idx < len(input_ids_list) - 1:
            # The verdict sequence is everything from '[' to the end of the trace
            verdict_indices = range(verdict_start_idx, len(input_ids_list))
            
            step_probabilities = []
            
            # 3. Calculate the probability the model assigned to each generated token
            for idx in verdict_indices:
                # The logits for the token at `idx` are produced by the model at `idx - 1`
                logits_for_step = outputs.logits[0, idx - 1] 
                
                # Convert logits to a probability distribution over the whole vocabulary
                probs = F.softmax(logits_for_step, dim=-1)

                predicted_token_id = torch.argmax(logits_for_step).item()
                # print(f"Sample {index}: Token index {idx}, Predicted token ID: {predicted_token_id}, Actual token ID: {input_ids_list[idx]}, Probability distribution shape: {probs.shape}")
                step_confidence = torch.max(probs).item()
                step_probabilities.append(step_confidence)
                
                # What token did the model actually generate?
                # actual_token_id = input_ids_list[idx]
                
                # # What was the probability assigned to that specific token?
                # token_prob = probs[actual_token_id].item()
                # step_probabilities.append(token_prob)
                
            # 4. Aggregate the confidence
            # Use the Geometric Mean to prevent long token sequences from penalizing the score
            if len(step_probabilities) > 0:
                product = np.prod(step_probabilities)
                llm_confidence = product ** (1.0 / len(step_probabilities))
            else:
                llm_confidence = 0.0
        else:
            # Fallback if no formatted verdict was found
            llm_confidence = 0.0

        # print(f"Sample {index}: LLM confidence in its answer: {llm_confidence:.4f}")
        # Store this value in your results dictionary
        results['llm_confidence'].append(llm_confidence)
    index += 1
with open("./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max512_temp0.0_all_logits.pkl", "wb") as outfile:
    outfile.write(pickle.dumps(results))

100%|██████████| 2100/2100 [13:35<00:00,  2.58it/s]


### Classifier

In [12]:
import json

def has_null(obj):
    """Returns True if the object or any of its children contain None."""
    if obj is None:
        return True
    if isinstance(obj, dict):
        return any(has_null(v) for v in obj.values())
    if isinstance(obj, list):
        return any(has_null(item) for item in obj)
    return False

def filter_null_containers(obj):
    """Removes elements entirely if they contain any null values."""
    if isinstance(obj, dict):
        return {k: filter_null_containers(v) for k, v in obj.items() if not has_null(v)}
    elif isinstance(obj, list):
        return [filter_null_containers(item) for item in obj if not has_null(item)]
    return obj

# Load, clean, and save
with open("./results/2100_samples_SATBench_Qwen3-8B/logical_result_max512_temp0.0_non-thinking.json", "r") as file:
    data = json.load(file)

clean_data = filter_null_containers(data)

with open("./results/2100_samples_SATBench_Qwen3-8B/logical_result_max512_temp0.0_non-thinking_removed_none.json", "w") as file:
    json.dump(clean_data, file, indent=4)


In [8]:
from model import *
import os
import torch
import numpy as np
from data_reader import load_hidden_states
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

classifier = SingleMLP_Classifier
results_file = "./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max128_temp0.0_verdict_only.pkl"
# classifier_model = classifier(4096).to(device)
# checkpoint = "./models/Llama-3.1-8B-Instruct/mlp_context_emb.pt"
# state_dict = torch.load(checkpoint, map_location=device)
# classifier_model.load_state_dict(state_dict)

Using device: cuda:0


In [9]:
def compute_confusion_stats(y_true, y_pred, labels=(0, 1)):
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    tn, fp, fn, tp = cm.ravel()
    stats = {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "accuracy": float((tp + tn) / cm.sum()),
        "precision": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
        "recall": float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
    }
    return stats, cm


def plot_confusion_matrix(cm, labels=("Hallucinated", "Non-hallucinated"), save_path="./results/confusion_matrix.png", label=None):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    ax.set_facecolor("white")

    cmap = plt.cm.Blues
    img = ax.imshow(cm, cmap=cmap, vmin=0, vmax=max(1, cm.max()))

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, color="#0f172a", fontsize=11)
    ax.set_yticklabels(labels, color="#0f172a", fontsize=11)
    ax.set_xlabel("Predicted label", color="#0f172a", fontsize=11)
    ax.set_ylabel("True label", color="#0f172a", fontsize=11)
    ax.set_title(f"{label} Confusion Matrix", color="#0f172a", fontsize=12, pad=10)
    ax.tick_params(axis="both", colors="#0f172a")
    for spine in ax.spines.values():
        spine.set_color("#1f2937")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            cell_color = cmap(img.norm(cm[i, j]))
            brightness = 0.299 * cell_color[0] + 0.587 * cell_color[1] + 0.114 * cell_color[2]
            text_color = "white" if brightness < 0.55 else "#111827"
            ax.text(j, i, format(cm[i, j], "d"),
                    ha="center", va="center",
                    color=text_color, fontsize=11, fontweight="bold")

    cbar = fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.yaxis.set_tick_params(color="#0f172a")
    plt.setp(cbar.ax.get_yticklabels(), color="#0f172a")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    # plt.close(fig)
    return save_path

In [10]:
from sklearn.model_selection import train_test_split
from tqdm import tqdm

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
batch_size = 128
learning_rate = 1e-4
weight_decay = 1e-2
save_model_dir = "./models/Llama3.1-8B"
save_name = "mlp_context_emb_SATBench.pt"
os.makedirs(save_model_dir, exist_ok=True)

results, correct = load_hidden_states(results_file)
labels = correct
pooled_embeddings = []
for i in range(len(results['generated_embeddings'])):
    embeddings = results['generated_embeddings'][i]
    emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
    pooled_emb = torch.mean(emb_sequence, dim=0)    # mean pooling of the embeddings for each sample
    # pooled_emb = torch.amax(emb_sequence, dim=0)    # max pooling
    pooled_embeddings.append(pooled_emb)
inputs = torch.stack(pooled_embeddings)
indices = range(len(inputs))

X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(inputs, labels.astype(int), indices, test_size=0.2, random_state=123)
classifier_model = classifier(X_train.shape[1]).to(device)
X_train = torch.tensor(X_train).to(device)
y_train = torch.tensor(y_train).to(torch.long).to(device)
X_test = torch.tensor(X_test).to(device)
y_test = torch.tensor(y_test).to(torch.long).to(device)

optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

for _ in tqdm(range(1001)):
    optimizer.zero_grad()
    sample = torch.randperm(X_train.shape[0])[:batch_size]
    pred = classifier_model(X_train[sample])
    loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
    loss.backward()
    optimizer.step()
# torch.save(classifier_model.state_dict(), f"{save_model_dir}/{save_name}")

classifier_model.eval()
with torch.no_grad():
    pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
    pred_probs = pred[:, 1].cpu().numpy()
    prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()

context_emb_roc, context_emb_acc = roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
print(f"Contextual Embeddings ROC AUC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")

Using device: cuda:0


/tmp/ipykernel_2061689/4120031513.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_2061689/4120031513.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:06<00:00, 162.42it/s]

Contextual Embeddings ROC AUC: 0.5023, Accuracy: 0.5071


In [11]:
pred_labels = prediction_classes.numpy()
true_labels = y_test.cpu().numpy()
cm = confusion_matrix(true_labels, pred_labels)
confusion_plot_path = plot_confusion_matrix(
    cm,
    labels=("Hallucinated", "Non-hallucinated"),
    save_path="./results/2100_samples_SATBench_Llama3.1-8B/verdict_only/confusion_matrix_hallu.png",
    label="Hallucinations Classifier"
)

In [5]:
print(test_indices)
print(len(test_indices))
# print(prediction_classes[0].item())

[1506, 1109, 297, 974, 1412, 613, 309, 1650, 1179, 50, 1651, 235, 1001, 887, 406, 147, 318, 1638, 679, 28, 776, 421, 981, 715, 1205, 1138, 817, 738, 1117, 104, 1421, 734, 1050, 474, 1334, 233, 967, 685, 1524, 932, 1322, 1515, 1589, 1440, 103, 1356, 1354, 1485, 1565, 1359, 308, 1309, 1360, 328, 189, 394, 1113, 1103, 1115, 928, 627, 783, 131, 1607, 521, 1646, 746, 722, 42, 1079, 304, 31, 1191, 1614, 600, 1397, 717, 811, 1315, 610, 1472, 1466, 1040, 35, 249, 993, 1184, 1166, 61, 598, 803, 276, 1180, 1364, 623, 1286, 167, 594, 1625, 944, 947, 646, 866, 1566, 1612, 1037, 558, 343, 1582, 772, 1323, 1391, 120, 961, 903, 1294, 725, 1149, 427, 651, 184, 704, 1170, 891, 156, 1632, 1028, 1002, 1473, 701, 1601, 319, 18, 1419, 260, 1018, 210, 1380, 1446, 666, 320, 1127, 182, 547, 1279, 996, 566, 63, 793, 472, 1547, 502, 1544, 1393, 1522, 1598, 1066, 1008, 499, 256, 316, 853, 1508, 1628, 564, 657, 842, 1498, 1564, 1636, 535, 1065, 252, 13, 1102, 1261, 1660, 261, 952, 812, 1207, 1209, 209, 134, 12, 4

In [6]:
from tqdm import tqdm
import json

pred_labels, true_labels = [], []
json_data = json.load(open("./results/2100_samples_SATBench_Qwen3-8B/logical_result_max512_temp0.0_non-thinking_removed_none.json", "r"))
for i in tqdm(range(len(test_indices))):
    id = test_indices[i]
    if json_data[id]["pred_label"]:
        if json_data[id]["pred_label"] == "SAT" and prediction_classes[i].item() == 1:
            pred_labels.append(1)
            if json_data[id]["correct_prediction"]:
                true_labels.append(1)
            else:
                true_labels.append(0)
        elif json_data[id]["pred_label"] == "UNSAT" and prediction_classes[i].item() == 1:
            pred_labels.append(0)
            if json_data[id]["correct_prediction"]:
                true_labels.append(0)
            else:
                true_labels.append(1)
        elif json_data[id]["pred_label"] == "SAT" and prediction_classes[i].item() == 0:
            pred_labels.append(0)
            if json_data[id]["correct_prediction"]:
                true_labels.append(1)
            else:
                true_labels.append(0)
        elif json_data[id]["pred_label"] == "UNSAT" and prediction_classes[i].item() == 0:
            pred_labels.append(1)
            if json_data[id]["correct_prediction"]:
                true_labels.append(0)
            else:
                true_labels.append(1)

100%|██████████| 333/333 [00:00<00:00, 119233.67it/s]


In [7]:
len(true_labels), len(pred_labels)

(333, 333)

In [8]:
# pred_labels = prediction_classes.numpy()
# true_labels = labels.cpu().numpy()
cm = confusion_matrix(true_labels, pred_labels)
confusion_plot_path = plot_confusion_matrix(
    cm,
    labels=("Unsatisfiable", "Satisfiable"),
    save_path="./results/2100_samples_SATBench_Llama3.1-8B/max_signal/confusion_matrix_reasoner.png",
    label="Hallucinations Classifier as Reasoner"
)

### Analysis on Non/Hallucination In/Correct + Confidence Value

In [7]:
from model import *
import os
import torch
import numpy as np
from data_reader import load_hidden_states
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib
from sklearn.model_selection import train_test_split
from tqdm import tqdm
matplotlib.use("Agg")
import matplotlib.pyplot as plt

results_file = "./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max512_temp0.0_v2.pkl"

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
batch_size = 128
learning_rate = 1e-4
weight_decay = 1e-2
save_model_dir = "./models/Llama3.1-8B"
# save_name = "mlp_context_emb_SATBench.pt"
os.makedirs(save_model_dir, exist_ok=True)

def compute_confusion_stats(y_true, y_pred, labels=(0, 1)):
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    tn, fp, fn, tp = cm.ravel()
    stats = {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "accuracy": float((tp + tn) / cm.sum()),
        "precision": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
        "recall": float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
    }
    return stats, cm


def plot_confusion_matrix(cm, labels=("Hallucinated", "Non-hallucinated"), save_path="./results/confusion_matrix.png", label=None):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    ax.set_facecolor("white")

    cmap = plt.cm.Blues
    img = ax.imshow(cm, cmap=cmap, vmin=0, vmax=max(1, cm.max()))

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, color="#0f172a", fontsize=11)
    ax.set_yticklabels(labels, color="#0f172a", fontsize=11)
    ax.set_xlabel("Predicted label", color="#0f172a", fontsize=11)
    ax.set_ylabel("True label", color="#0f172a", fontsize=11)
    ax.set_title(f"{label} Confusion Matrix", color="#0f172a", fontsize=12, pad=10)
    ax.tick_params(axis="both", colors="#0f172a")
    for spine in ax.spines.values():
        spine.set_color("#1f2937")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            cell_color = cmap(img.norm(cm[i, j]))
            brightness = 0.299 * cell_color[0] + 0.587 * cell_color[1] + 0.114 * cell_color[2]
            text_color = "white" if brightness < 0.55 else "#111827"
            ax.text(j, i, format(cm[i, j], "d"),
                    ha="center", va="center",
                    color=text_color, fontsize=11, fontweight="bold")

    cbar = fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.yaxis.set_tick_params(color="#0f172a")
    plt.setp(cbar.ax.get_yticklabels(), color="#0f172a")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    # plt.close(fig)
    return save_path

def gen_classifier_roc(inputs, labels, classifier, save_name="classifier_model.pt"):
    X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(inputs, labels.astype(int), indices, test_size=0.2, random_state=123)
    classifier_model = classifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    # torch.save(classifier_model.state_dict(), f"{save_model_dir}/{save_name}")

    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        pred_probs = pred[:, 1].cpu().numpy()
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        context_emb_roc, context_emb_acc = roc_auc_score(y_test.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==y_test.cpu().numpy()).mean()

    pred_labels = prediction_classes.numpy()
    true_labels = y_test.cpu().numpy()
    cm = confusion_matrix(true_labels, pred_labels)
    confusion_plot_path = plot_confusion_matrix(
        cm,
        labels=("Hallucinated", "Non-hallucinated"),
        save_path="./results/2100_samples_SATBench_Llama3.1-8B/category_analysis/confusion_matrix_hallu.png",
        label="Hallucinations Classifier"
    )
    print(f"Contextual Embeddings ROC AUC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")

    return pred_probs, context_emb_roc, context_emb_acc, prediction_classes, y_test.cpu().numpy(), test_indices
# print(f"Contextual Embeddings ROC AUC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")

Using device: cuda:0


In [6]:
import pandas as pd

def analyze_hallucination_categories(pred_probs, predictions, llm_correct, save_csv_path="./results/2100_samples_SATBench_Llama3.1-8B/category_analysis/category_analysis.csv"):
    """
    Categorizes classifier outputs into 3 logic buckets and computes confidence values.
    
    Args:
        pred_probs: Continuous probability of hallucination (from classifier)
        predictions: Binary predicted class (0 = Non-Hallucinated, 1 = Hallucinated)
        llm_correct: Ground truth LLM answer correctness (0 = Incorrect, 1 = Correct)
    """
    # Create a DataFrame for easy slicing
    df = pd.DataFrame({
        'llm_correct': llm_correct,
        'classifier_pred': predictions,
        'hallucination_prob': pred_probs
    })
    
    # Calculate Confidence: 
    # If the classifier predicts Hallucinated (1), confidence is the prob of 1.
    # If it predicts Non-Hallucinated (0), confidence is the prob of 0 (1 - prob of 1).
    df['confidence'] = np.where(df['classifier_pred'] == 1, 
                                df['hallucination_prob'], 
                                1.0 - df['hallucination_prob'])

    # Define the categories based on your criteria
    conditions = [
        (df['llm_correct'] == 1) & (df['classifier_pred'] == 1), # 1: Non-Hallucinated Correct
        (df['llm_correct'] == 0) & (df['classifier_pred'] == 0), # 2: Hallucinated Correct
        (df['llm_correct'] == 1) & (df['classifier_pred'] == 0), # 3: Hallucinated Incorrect
        (df['llm_correct'] == 0) & (df['classifier_pred'] == 1)  # 4: Non-Hallucinated Incorrect
    ]
    
    choices = [
        '1_Non-Hallucinated_Correct',
        '2_Hallucinated_Correct',
        '3_Hallucinated_Incorrect',
        '4_Non-Hallucinated_Incorrect'
    ]
    
    df['category'] = np.select(conditions, choices, default='Unknown')

    # Group by category and calculate statistics
    summary = df.groupby('category').agg(
        count=('category', 'size'),
        mean_confidence=('confidence', 'mean'),
        median_confidence=('confidence', 'median'),
        std_confidence=('confidence', 'std'),
        min_confidence=('confidence', 'min'),
        max_confidence=('confidence', 'max')
    ).reset_index()

    print("\n=== Hallucination Classifier Category Analysis ===")
    print(summary.to_string(index=False))
    
    # Save the detailed trace-by-trace breakdown for manual inspection
    os.makedirs(os.path.dirname(save_csv_path) or ".", exist_ok=True)
    summary.to_csv(save_csv_path, index=False)
    print(f"\nDetailed analysis saved to: {save_csv_path}")
    
    return df, summary

# --- Implementation in your execution block ---
# After capturing the outputs from your updated gen_classifier_roc:
try:
    classifier = SingleMLP_Classifier
    results, correct = load_hidden_states(results_file)
    labels = correct
    if results['generated_embeddings'][0].ndim == 1:
        inputs = torch.stack([torch.tensor(emb, dtype=torch.float32) for emb in results['generated_embeddings']])
    else:
        pooled_embeddings = []
        for i in range(len(results['generated_embeddings'])):
            embeddings = results['generated_embeddings'][i]
            emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
            pooled_emb = torch.mean(emb_sequence, dim=0)    # mean pooling of the embeddings for each sample
            # pooled_emb = torch.amax(emb_sequence, dim=0)    # max pooling
            pooled_embeddings.append(pooled_emb)
        inputs = torch.stack(pooled_embeddings)
    indices = range(len(inputs))
    # Notice we catch pred_probs now
    pred_probs, context_emb_roc, context_emb_acc, prediction_classes, y_true_test, test_indices = gen_classifier_roc(inputs, labels, classifier)
    
    # Run the analysis
    df_analysis, summary_stats = analyze_hallucination_categories(
        pred_probs=pred_probs,
        predictions=prediction_classes,
        llm_correct=y_true_test
    )
    
except Exception as err:
    print(err)

/tmp/ipykernel_2007739/2129554811.py:81: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_2007739/2129554811.py:83: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:05<00:00, 183.64it/s]


Contextual Embeddings ROC AUC: 0.5820, Accuracy: 0.5706

=== Hallucination Classifier Category Analysis ===
                    category  count  mean_confidence  median_confidence  std_confidence  min_confidence  max_confidence
  1_Non-Hallucinated_Correct    137         0.690599           0.662944        0.108736        0.516530        0.955786
      2_Hallucinated_Correct     57         0.668942           0.645134        0.114432        0.502519        0.947638
    3_Hallucinated_Incorrect     46         0.671890           0.653163        0.122405        0.504625        0.942934
4_Non-Hallucinated_Incorrect    100         0.669807           0.643612        0.109677        0.503842        0.967005

Detailed analysis saved to: ./results/2100_samples_SATBench_Llama3.1-8B/category_analysis/category_analysis.csv


In [8]:
import pandas as pd

def analyze_hallucination_categories(pred_probs, predictions, llm_correct, save_csv_path="./results/2100_samples_SATBench_Llama3.1-8B/category_analysis/category_analysis.csv"):
    """
    Categorizes classifier outputs into 3 logic buckets and computes confidence values.
    
    Args:
        pred_probs: Continuous probability of hallucination (from classifier)
        predictions: Binary predicted class (0 = Non-Hallucinated, 1 = Hallucinated)
        llm_correct: Ground truth LLM answer correctness (0 = Incorrect, 1 = Correct)
    """
    # Create a DataFrame for easy slicing
    df = pd.DataFrame({
        'llm_correct': llm_correct,
        'classifier_pred': predictions,
        'hallucination_prob': pred_probs
    })
    
    # Calculate Confidence: 
    # If the classifier predicts Hallucinated (1), confidence is the prob of 1.
    # If it predicts Non-Hallucinated (0), confidence is the prob of 0 (1 - prob of 1).
    df['confidence'] = np.where(df['classifier_pred'] == 1, 
                                df['hallucination_prob'], 
                                1.0 - df['hallucination_prob'])

    # Define the categories based on your criteria
    conditions = [
        (df['llm_correct'] == 1) & (df['classifier_pred'] == 1), # 1: Non-Hallucinated Correct
        (df['llm_correct'] == 0) & (df['classifier_pred'] == 0), # 2: Hallucinated Correct
        (df['llm_correct'] == 1) & (df['classifier_pred'] == 0), # 3: Hallucinated Incorrect
        (df['llm_correct'] == 0) & (df['classifier_pred'] == 1)  # 4: Non-Hallucinated Incorrect
    ]
    
    choices = [
        '1_Non-Hallucinated_Correct',
        '2_Hallucinated_Correct',
        '3_Hallucinated_Incorrect',
        '4_Non-Hallucinated_Incorrect'
    ]
    
    df['category'] = np.select(conditions, choices, default='Unknown')

    # Group by category and calculate statistics
    summary = df.groupby('category').agg(
        count=('category', 'size'),
        mean_confidence=('confidence', 'mean'),
        median_confidence=('confidence', 'median'),
        std_confidence=('confidence', 'std'),
        min_confidence=('confidence', 'min'),
        max_confidence=('confidence', 'max')
    ).reset_index()

    print("\n=== Hallucination Classifier Category Analysis ===")
    print(summary.to_string(index=False))
    
    # Save the detailed trace-by-trace breakdown for manual inspection
    os.makedirs(os.path.dirname(save_csv_path) or ".", exist_ok=True)
    summary.to_csv(save_csv_path, index=False)
    print(f"\nDetailed analysis saved to: {save_csv_path}")
    
    return df, summary

# --- Implementation in your execution block ---
# After capturing the outputs from your updated gen_classifier_roc:
try:
    classifier = SingleMLP_Classifier
    results, correct = load_hidden_states(results_file)
    labels = correct
    if results['generated_embeddings'][0].ndim == 1:
        inputs = torch.stack([torch.tensor(emb, dtype=torch.float32) for emb in results['generated_embeddings']])
    else:
        pooled_embeddings = []
        for i in range(len(results['generated_embeddings'])):
            embeddings = results['generated_embeddings'][i]
            emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
            pooled_emb = torch.mean(emb_sequence, dim=0)    # mean pooling of the embeddings for each sample
            # pooled_emb = torch.amax(emb_sequence, dim=0)    # max pooling
            pooled_embeddings.append(pooled_emb)
        inputs = torch.stack(pooled_embeddings)
    indices = range(len(inputs))
    # Notice we catch pred_probs now
    pred_probs, context_emb_roc, context_emb_acc, prediction_classes, y_true_test, test_indices = gen_classifier_roc(inputs, labels, classifier)
    
    # Run the analysis
#     df_analysis, summary_stats = analyze_hallucination_categories(
#         pred_probs=pred_probs,
#         predictions=prediction_classes,
#         llm_correct=y_true_test
#     )
    
except Exception as err:
    print(err)

/tmp/ipykernel_2045781/2129554811.py:81: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_2045781/2129554811.py:83: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:05<00:00, 183.02it/s]


Contextual Embeddings ROC AUC: 0.5784, Accuracy: 0.5794


In [9]:
from tqdm import tqdm
import json
import pandas as pd
import numpy as np
import pickle

pred_labels, true_labels, confidences = [], [], []
# 1. Load the extracted results containing the LLM's confidence scores
results_file = "./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max512_temp0.0_all_logits.pkl"
with open(results_file, "rb") as f:
    results = pickle.load(f)

for i in tqdm(range(len(test_indices))):
    id = test_indices[i]
    pred_labels.append(results['llm_prediction'][id])
    true_labels.append(results['ground_truth'][id])
    confidences.append(results['llm_confidence'][id])

# Ensure lengths match. Assuming you have a list/array of trace validity labels 
# (1 = Hallucinated/Flawed Trace, 0 = Non-Hallucinated/Sound Trace). 
# Replace `trace_hallucinated_labels` with your actual oracle labels or classifier predictions.
llm_pred = np.array(pred_labels)
ground_truth = np.array(true_labels)
llm_confidence = np.array(confidences)

# 2. Build the DataFrame
df = pd.DataFrame({
    'ground_truth': ground_truth,
    'llm_pred': llm_pred,
    'llm_confidence': llm_confidence
})

# Filter out samples where the LLM failed to output a verifiable bracket/confidence score
df = df[df['llm_confidence'] > 0.0].copy()

# 3. Categorize the outcomes
conditions = [
    (df['ground_truth'] == 1) & (df['llm_pred'] == 1),
    (df['ground_truth'] == 0) & (df['llm_pred'] == 0),
    (df['ground_truth'] == 1) & (df['llm_pred'] == 0),
    (df['ground_truth'] == 0) & (df['llm_pred'] == 1)
]

choices = [
    '1_SAT_Correct',
    '2_UNSAT_Correct',
    '3_UNSAT_Incorrect',
    '4_SAT_Incorrect'
]

df['category'] = np.select(conditions, choices, default='Unknown')

# 4. Compute statistics for the categories
summary = df.groupby('category').agg(
    count=('category', 'size'),
    mean_confidence=('llm_confidence', 'mean'),
    median_confidence=('llm_confidence', 'median'),
    std_confidence=('llm_confidence', 'std'),
    min_confidence=('llm_confidence', 'min'),
    max_confidence=('llm_confidence', 'max')
).reset_index()

# 5. Format and display
summary['mean_confidence'] = summary['mean_confidence'].round(4)
summary['median_confidence'] = summary['median_confidence'].round(4)
summary['std_confidence'] = summary['std_confidence'].round(4)

print("\n=== LLM Autoregressive Confidence by Hallucination Category ===")
print(summary.to_string(index=False))

# Optional: Save to CSV for plotting
summary.to_csv("./results/2100_samples_SATBench_Llama3.1-8B/category_analysis/llm_confidence_all_logits.csv", index=False)

100%|██████████| 340/340 [00:00<00:00, 513526.60it/s]


=== LLM Autoregressive Confidence by Hallucination Category ===
         category  count  mean_confidence  median_confidence  std_confidence  min_confidence  max_confidence
    1_SAT_Correct    117           0.8593             0.8713          0.0589        0.637743        0.968071
  2_UNSAT_Correct     66           0.8079             0.8116          0.0543        0.677475        0.932299
3_UNSAT_Incorrect     42           0.8164             0.8168          0.0511        0.728999        0.932920
  4_SAT_Incorrect    115           0.8650             0.8755          0.0568        0.711426        0.959144


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plot_dir = Path("./results/2100_samples_SATBench_Llama3.1-8B/category_analysis")
df_llm = pd.read_csv(plot_dir / "llm_confidence_verdict_logits.csv")
df_cat = pd.read_csv(plot_dir / "category_analysis.csv")

# Extract means for the bars
llm_means = df_llm['mean_confidence'].values
cat_means = df_cat['mean_confidence'].values

# Clean up categories for X-axis
categories = [
    "SAT / Non-Hallucinated\nCorrect", 
    "UNSAT / Hallucinated\nCorrect", 
    "UNSAT / Hallucinated\nIncorrect", 
    "SAT / Non-Hallucinated\nIncorrect"
]

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')

x = np.arange(len(categories))
width = 0.3  # Width of the grouped bars

# Colors mapping to the example chart
color_a = '#4472C4' # Classic Blue
color_b = '#ED7D31' # Classic Orange

# Plot the bars
rects1 = ax.bar(x - width/2, llm_means, width, label='LLM Confidence', color=color_a)
rects2 = ax.bar(x + width/2, cat_means, width, label='Classifier Confidence', color=color_b)

# Title and axis labels
ax.set_title('Confidence Value Comparison', fontsize=20, pad=20, color='black')
ax.set_ylabel('Mean Confidence', fontsize=14, color='black')
# ax.set_xlabel('Hallucination State', fontsize=14, color='black', labelpad=10)

# X-axis ticks
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11, color='black')
ax.tick_params(axis='x', length=0, pad=10)

# Y-axis ticks and limits
ax.set_ylim(0, 1.2) # Extended to give room for the data labels on top
ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2])
ax.tick_params(axis='y', labelsize=11, colors='black', length=0)

# Add light horizontal gridlines behind the bars
ax.yaxis.grid(True, color='#E0E0E0', linestyle='-', linewidth=1)
ax.set_axisbelow(True)

# Remove the top, right, and left spines to match the clean example style
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_color('#CCCCCC')

# Add value labels precisely on top of the bars
def autolabel(rects):
    """Attach a text label above each bar, displaying its height."""
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5),  # 5 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, color='black')

autolabel(rects1)
autolabel(rects2)

# Format the legend at the bottom, horizontally, with square markers
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), 
            ncol=2, frameon=False, fontsize=12, handlelength=1.2, handleheight=1.2)

plt.tight_layout()
plt.savefig(plot_dir / "confidence_comparison1.png", dpi=150, bbox_inches='tight')

### BiGRU Verifier and Attention-Pool

In [11]:
from model import *
import os
import torch
import numpy as np
from data_reader import load_hidden_states
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

classifier = BiGRUAttention_Classifier
results_file = "./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max512_temp0.0_delimiter_steps_layer21.pkl"
# classifier_model = classifier(4096).to(device)
# checkpoint = "./models/Llama-3.1-8B-Instruct/mlp_context_emb.pt"
# state_dict = torch.load(checkpoint, map_location=device)
# classifier_model.load_state_dict(state_dict)

Using device: cuda:0


In [12]:
batch_size = 128
learning_rate = 1e-4
weight_decay = 1e-2

def gen_classifier_roc(X_padded, lengths, labels, model_class, save_name="classifier_model.pt"):
    # Split using indices so we keep X, lengths, and labels aligned
    indices = np.arange(len(labels))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=123)
    
    X_train = X_padded[train_idx].to(device)
    lengths_train = lengths[train_idx].to(device)
    y_train = torch.tensor(labels[train_idx], dtype=torch.long, device=device)
    
    X_test = X_padded[test_idx].to(device)
    lengths_test = lengths[test_idx].to(device)
    y_test = torch.tensor(labels[test_idx], dtype=torch.long, device=device)

    # Initialize the new sequence model
    classifier_model = model_class(input_dim=X_train.shape[2]).to(device)
    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    classifier_model.train()
    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample_idx = torch.randperm(X_train.shape[0])[:batch_size]
        
        pred = classifier_model(X_train[sample_idx], lengths_train[sample_idx])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample_idx])
        loss.backward()
        optimizer.step()
        
    # torch.save(classifier_model.state_dict(), f"{save_model_dir}/{save_name}")

    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test, lengths_test), dim=1)
        prediction_classes = (pred[:, 1] > 0.5).type(torch.long).cpu()
        
        roc = roc_auc_score(y_test.cpu(), pred[:, 1].cpu())
        acc = (prediction_classes.numpy() == y_test.cpu().numpy()).mean()
        return roc, acc, prediction_classes.numpy(), y_test.cpu().numpy()

In [13]:
all_results = {}
# try:
classifier_results = {}
print(f"Loading data from {results_file}")

# Assuming load_hidden_states returns the dict and the correct labels list
results, correct_list = load_hidden_states(results_file)

# Extract variable-length sequences and their lengths
sequence_tensors = []
lengths_list = []

for i in range(len(results['generated_embeddings'])):
    embeddings = results['generated_embeddings'][i]
    
    # If a trace had 0 steps (parsing failure), pad with a zero-vector length 1
    if embeddings.shape[0] == 0:
        embeddings = np.zeros((1, embeddings.shape[1]), dtype=np.float32)
        
    emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
    sequence_tensors.append(emb_sequence)
    lengths_list.append(emb_sequence.shape[0])

# Pad sequences to max length in the dataset -> Shape: [B, T_max, hidden_size]
padded_embeddings_tensor = torch.nn.utils.rnn.pad_sequence(
    sequence_tensors, 
    batch_first=True
)
lengths_tensor = torch.tensor(lengths_list, dtype=torch.long)
correct_array = np.array(correct_list)
# print(f"Embeddings shape: {results['generated_embeddings'][0].shape}")
# print(f"Embedding 0: {results['generated_embeddings'][0]}")
# print(f"Padded embeddings shape: {padded_embeddings_tensor.shape}, Lengths shape: {lengths_tensor.shape}, Labels shape: {correct_array.shape}")
# Train and Evaluate
context_emb_roc, context_emb_acc, prediction_classes, y_true = gen_classifier_roc(
    X_padded=padded_embeddings_tensor, 
    lengths=lengths_tensor, 
    labels=correct_array, 
    model_class=classifier, 
    save_name="bigru_attention_context_emb.pt"
)

print(f"Contextual Embeddings ROC AUC: {context_emb_roc:.4f},\n Accuracy: {context_emb_acc:.4f}")

    # confusion_stats, confusion_matrix_values = compute_confusion_stats(y_true, prediction_classes, labels=(0, 1))
    # confusion_plot_path = plot_confusion_matrix(
    #     confusion_matrix_values,
    #     labels=("Hallucinated", "Non-hallucinated"),
    #     save_path="./results/2100_samples_SATBench_Llama3.1-8B/confusion_matrix_hallu.png"
    # )

    # classifier_results['context_emb_roc'] = context_emb_roc
    # classifier_results['context_emb_acc'] = context_emb_acc
    # classifier_results['TP'] = confusion_stats['TP']
    # classifier_results['TN'] = confusion_stats['TN']
    # classifier_results['FP'] = confusion_stats['FP']
    # classifier_results['FN'] = confusion_stats['FN']
    
    # all_results[results_file] = classifier_results.copy()
    
# except Exception as err:
#     print(err)

Loading data from ./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max512_temp0.0_delimiter_steps_layer21.pkl


100%|██████████| 1001/1001 [01:07<00:00, 14.76it/s]

Contextual Embeddings ROC AUC: 0.5614,
 Accuracy: 0.5526


### Dual-Stream Relational Probe

In [1]:
import os
import re
import json
import pickle
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Optional

import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---------------------------------------------------------------------------
# Configuration & Model Loading
# ---------------------------------------------------------------------------
model_dir = Path("./.cache/models")
model_name = "meta-llama/Llama-3.1-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    # load_in_8bit=True,
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=model_dir,
    trust_remote_code=True
)

# CRITICAL: attn_implementation="eager" is required to populate outputs.attentions
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=model_dir,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    # quantization_config=quantization_config,
    attn_implementation="eager"
)
model.eval()

# Probing layer index (-12 is layer 21 of 32 in Llama-3.1-8B)
PROBE_LAYER = -1

system_prompt = """You are a logical reasoning assistant. You are given a logic puzzle.
Guidelines:
- All constraints come **only** from the <conditions> section.
- The <scenario> provides background and intuition, but **does not impose any additional rules or constraints**.
- All variables represent **independent decisions**; there is no mutual exclusivity or implicit linkage unless stated explicitly in <conditions>.
- Variables not mentioned in <conditions> are considered unknown and irrelevant to satisfiability.
Your task:
- If the puzzle is satisfiable, propose one valid assignment that satisfies all the conditions.
- If the puzzle is unsatisfiable, explain why some of the conditions cannot all be true at once.
Think step by step but don't return any intermediate reasoning, only give a brief response. At the end of your response, output exactly one of the following labels on a new line:
[SAT]- if a valid assignment exists
[UNSAT]- if the constraints cannot be satisfied
Do not add any text or formatting after the final label.
"""

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
# ---------------------------------------------------------------------------
# Helper Functions: Span Extraction & Feature Math
# ---------------------------------------------------------------------------
def split_trace_into_spans(trace: str):
    verdict_match = re.search(r"\s*\[(SAT|UNSAT)\]\s*$", trace)
    reasoning = trace[:verdict_match.start()] if verdict_match else trace
    lines = [
        (match.group().strip(), match.start(), match.end())
        for match in re.finditer(r"[^\n]+", reasoning)
        if match.group().strip()
    ]
    if len(lines) > 1:
        return reasoning, lines

    sentences = [
        (match.group().strip(), match.start(), match.end())
        for match in re.finditer(r".+?(?:[.!?](?=\s|$)|$)", reasoning.strip())
        if match.group().strip()
    ]
    return reasoning, sentences


def find_token_span(haystack: List[int], needle: List[int], max_search_idx: int) -> Optional[Tuple[int, int]]:
    """Locates the start and end token indices of a clause within the prompt region."""
    len_n = len(needle)
    if len_n == 0:
        return None
    search_limit = min(len(haystack) - len_n + 1, max_search_idx)
    for i in range(search_limit):
        if haystack[i : i + len_n] == needle:
            return i, i + len_n
    # Boundary fallback (strips boundary token in case of whitespace/punctuation shift)
    if len_n > 2:
        for i in range(search_limit):
            if haystack[i : i + len_n - 2] == needle[1:-1]:
                return max(0, i - 1), min(len(haystack), i + len_n - 1)
    return None


# def compute_global_coverage_features(
#     M: torch.Tensor,           # Shape: [K, T]
#     step_embeds: torch.Tensor  # Shape: [T, d_model]
# ) -> np.ndarray:
#     """Computes the fixed 12-dimensional spectral coverage & trajectory vector."""
#     eps = 1e-8
#     K, T = M.shape

#     if K == 0 or T == 0:
#         return np.zeros(12, dtype=np.float32)

#     # 1. Clause coverage stats across K clauses
#     clause_max_attn = M.max(dim=1).values  # [K]
#     clause_mean_attn = M.mean(dim=1)       # [K]

#     min_cov = clause_max_attn.min()
#     q10_cov = torch.quantile(clause_max_attn, 0.10) if K > 1 else clause_max_attn[0]
#     q25_cov = torch.quantile(clause_max_attn, 0.25) if K > 1 else clause_max_attn[0]
#     mean_cov = clause_max_attn.mean()
#     std_cov = clause_max_attn.std(unbiased=False) if K > 1 else torch.tensor(0.0)

#     # 2. Clause attention entropy (detects if model fixates on only 1 constraint)
#     p_clause = clause_mean_attn / (clause_mean_attn.sum() + eps)
#     denom = torch.log(torch.tensor(float(K))) if K > 1 else torch.tensor(1.0)
#     clause_entropy = -(p_clause * torch.log(p_clause + eps)).sum() / (denom + eps)

#     # 3. Spectral features: Top-4 normalized singular values of M
#     singular_vals = torch.linalg.svdvals(M)
#     singular_vals = singular_vals / (singular_vals.sum() + eps)
#     top4_sv = torch.zeros(4, dtype=torch.float32)
#     num_sv = min(4, len(singular_vals))
#     top4_sv[:num_sv] = singular_vals[:num_sv]

#     # 4. Length-corrected trajectory directness in activation space
#     if T > 1:
#         step_diffs = step_embeds[1:] - step_embeds[:-1]
#         path_length = torch.norm(step_diffs, p=2, dim=-1).sum()
#         net_displacement = torch.norm(step_embeds[-1] - step_embeds[0], p=2)
#         raw_directness = net_displacement / (path_length + eps)
#         corrected_directness = raw_directness * (float(T) ** 0.5)
#         mean_step_cos = torch.nn.functional.cosine_similarity(
#             step_embeds[1:], step_embeds[:-1], dim=-1
#         ).mean()
#     else:
#         corrected_directness = torch.tensor(1.0)
#         mean_step_cos = torch.tensor(1.0)

#     vec = torch.stack([
#         min_cov, q10_cov, q25_cov, mean_cov, std_cov,
#         clause_entropy,
#         top4_sv[0], top4_sv[1], top4_sv[2], top4_sv[3],
#         corrected_directness, mean_step_cos
#     ])
#     return vec.detach().cpu().numpy().astype(np.float32)

def compute_global_coverage_features(
    M: torch.Tensor,           # Shape: [K, T]
    step_embeds: torch.Tensor  # Shape: [T, d_model]
) -> np.ndarray:
    """Computes the fixed 12-dimensional spectral coverage & trajectory vector."""
    eps = 1e-8
    K, T = M.shape
    
    # Grab the active device (e.g., cuda:0) to prevent CPU/GPU mismatch
    device = M.device  

    if K == 0 or T == 0:
        return np.zeros(12, dtype=np.float32)

    # 1. Clause coverage stats across K clauses
    clause_max_attn = M.max(dim=1).values  # [K]
    clause_mean_attn = M.mean(dim=1)       # [K]

    min_cov = clause_max_attn.min()
    q10_cov = torch.quantile(clause_max_attn, 0.10) if K > 1 else clause_max_attn[0]
    q25_cov = torch.quantile(clause_max_attn, 0.25) if K > 1 else clause_max_attn[0]
    mean_cov = clause_max_attn.mean()
    
    # FIX: Ensure std_cov fallback is placed on the correct device
    std_cov = clause_max_attn.std(unbiased=False) if K > 1 else torch.tensor(0.0, device=device)

    # 2. Clause attention entropy (detects if model fixates on only 1 constraint)
    p_clause = clause_mean_attn / (clause_mean_attn.sum() + eps)
    
    # FIX: Ensure denom is on the correct device
    denom = torch.log(torch.tensor(float(K), device=device)) if K > 1 else torch.tensor(1.0, device=device)
    clause_entropy = -(p_clause * torch.log(p_clause + eps)).sum() / (denom + eps)

    # 3. Spectral features: Top-4 normalized singular values of M
    singular_vals = torch.linalg.svdvals(M)
    singular_vals = singular_vals / (singular_vals.sum() + eps)
    
    # FIX: Ensure the zero vector is initialized on the GPU
    top4_sv = torch.zeros(4, dtype=torch.float32, device=device)
    num_sv = min(4, len(singular_vals))
    top4_sv[:num_sv] = singular_vals[:num_sv]

    # 4. Length-corrected trajectory directness in activation space
    if T > 1:
        step_diffs = step_embeds[1:] - step_embeds[:-1]
        path_length = torch.norm(step_diffs, p=2, dim=-1).sum()
        net_displacement = torch.norm(step_embeds[-1] - step_embeds[0], p=2)
        raw_directness = net_displacement / (path_length + eps)
        corrected_directness = raw_directness * (float(T) ** 0.5)
        mean_step_cos = torch.nn.functional.cosine_similarity(
            step_embeds[1:], step_embeds[:-1], dim=-1
        ).mean()
    else:
        # FIX: Ensure fallbacks are on the GPU
        corrected_directness = torch.tensor(1.0, device=device)
        mean_step_cos = torch.tensor(1.0, device=device)

    # Stack the 12 features (all now guaranteed to be on the same device)
    vec = torch.stack([
        min_cov, q10_cov, q25_cov, mean_cov, std_cov,
        clause_entropy,
        top4_sv[0], top4_sv[1], top4_sv[2], top4_sv[3],
        corrected_directness, mean_step_cos
    ])
    
    return vec.detach().cpu().numpy().astype(np.float32)


# ---------------------------------------------------------------------------
# Main Extraction Loop
# ---------------------------------------------------------------------------
input_json = "./results/2100_samples_SATBench_Llama3.1-8B/logical_result_max512_temp0.0_v1.json"
output_pkl = "./results/400_samples_SATBench_Llama3.1-8B/dual_stream_features_layer31.pkl"
# os.makedirs(Path(output_pkl).parent, exist_ok=True)

data = json.load(open(input_json, "r"))
results = defaultdict(list)
hidden_size = model.config.hidden_size

for index, item in enumerate(tqdm(data[:400])):
    if not (item["pred_label"] and len(item["model_trace"].strip()) > 7):
        continue

    scenario = item["scenario"]
    conditions = item["conditions"]
    question = item["question"]
    trace = item["model_trace"]

    user_prompt = f"<scenario>\n{scenario}\n<conditions>\n{conditions}\n<question>\n{question}\n"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    full_messages = messages + [{"role": "assistant", "content": trace}]

    # Determine prompt token length
    tokenized_query = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    prompt_length = tokenized_query.input_ids.shape[-1]

    # Full input tokens
    full_inputs = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).input_ids.to(model.device)

    # 1. Forward pass extracting target hidden states and attention matrices
    with torch.no_grad():
        outputs = model(
            input_ids=full_inputs,
            use_cache=False,
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True
        )

    # Extract hidden states at the selected probing layer: [seq_len, d_model]
    h_layer = outputs.hidden_states[PROBE_LAYER][0].to(torch.float32)
    # Extract attention weights at the probing layer: [num_heads, seq_len, seq_len]
    attn_layer = outputs.attentions[PROBE_LAYER][0].to(torch.float32)

    # -----------------------------------------------------------------------
    # Step A: Identify Reasoning Step Token Spans
    # -----------------------------------------------------------------------
    reasoning, step_spans = split_trace_into_spans(trace)
    trace_start = prompt_length - 1
    previous_token_end = 0
    step_token_spans: List[Tuple[int, int]] = []
    step_embeddings_list: List[torch.Tensor] = []

    for _, _, step_end_char in step_spans:
        cumulative_text = reasoning[:step_end_char]
        cumulative_token_end = len(tokenizer(cumulative_text, add_special_tokens=False).input_ids)

        step_start = trace_start + previous_token_end
        step_stop = trace_start + cumulative_token_end

        if step_stop > step_start:
            delimiter_idx = step_stop - 1
            step_embeddings_list.append(h_layer[delimiter_idx])
            step_token_spans.append((step_start, step_stop))

        previous_token_end = cumulative_token_end

    # -----------------------------------------------------------------------
    # Step B: Identify Condition Clause Spans in the Prompt
    # -----------------------------------------------------------------------
    if isinstance(conditions, list):
        clause_texts = [str(c).strip() for c in conditions if str(c).strip()]
    else:
        clause_texts = [line.strip() for line in str(conditions).strip().split("\n") if line.strip()]

    input_ids_list = full_inputs[0].tolist()
    clause_token_spans: List[Tuple[int, int]] = []
    clause_embeddings_list: List[torch.Tensor] = []

    for c_text in clause_texts:
        c_tokens = tokenizer(c_text, add_special_tokens=False).input_ids
        span = find_token_span(input_ids_list, c_tokens, max_search_idx=prompt_length)
        if span is not None:
            c_start, c_end = span
            clause_token_spans.append((c_start, c_end))
            # Clause embedding is the mean of its constituent token states
            clause_embeddings_list.append(h_layer[c_start:c_end].mean(dim=0))

    # -----------------------------------------------------------------------
    # Step C: Construct the 3 Dual-Stream Tensors
    # -----------------------------------------------------------------------
    K = len(clause_token_spans)
    T = len(step_token_spans)

    if K > 0 and T > 0:
        # Tensor 1: step_embeds [T, d_model]
        step_embeds = torch.stack(step_embeddings_list, dim=0)
        clause_embeds = torch.stack(clause_embeddings_list, dim=0)  # [K, d_model]

        # Compute Clause-to-Step Attention Matrix M [K, T]
        M = torch.zeros((K, T), device=h_layer.device, dtype=torch.float32)
        for k, (c_start, c_end) in enumerate(clause_token_spans):
            for t, (s_start, s_end) in enumerate(step_token_spans):
                # Attention sub-block: queries in Step t -> keys in Clause k
                sub_attn = attn_layer[:, s_start:s_end, c_start:c_end]  # [num_heads, |S_t|, |C_k|]
                # Sum over clause key tokens, mean over step query tokens, max over heads
                head_scores = sub_attn.sum(dim=-1).mean(dim=-1)         # [num_heads]
                M[k, t] = head_scores.max()

        # Tensor 2: premise_contexts [T, d_model]
        # Column-normalize M so each step t yields a distribution over the K clauses
        M_col_norm = M / (M.sum(dim=0, keepdim=True) + 1e-8)  # [K, T]
        premise_contexts = torch.matmul(M_col_norm.T, clause_embeds)  # [T, d_model]

        # Tensor 3: cov_features [12]
        cov_features = compute_global_coverage_features(M, step_embeds)

        step_embeds_np = step_embeds.detach().cpu().numpy()
        premise_contexts_np = premise_contexts.detach().cpu().numpy()
    else:
        # Graceful fallback for empty traces or unparsed constraints
        step_embeds_np = np.zeros((1, hidden_size), dtype=np.float32)
        premise_contexts_np = np.zeros((1, hidden_size), dtype=np.float32)
        cov_features = np.zeros(12, dtype=np.float32)

    # -----------------------------------------------------------------------
    # Store results
    # -----------------------------------------------------------------------
    results['generated_embeddings'].append(step_embeds_np)      # [T, d_model]
    results['premise_contexts'].append(premise_contexts_np)      # [T, d_model]
    results['cov_features'].append(cov_features)                # [12]
    results['correct'].append(int(item["correct_prediction"]))
    results['index'].append(index)

    # print(f"Processed sample {index}: {len(step_token_spans)} steps, {len(clause_token_spans)} clauses.")
    # print(f"Step embeddings shape: {step_embeds_np.shape}, Premise contexts shape: {premise_contexts_np.shape}, Coverage features shape: {cov_features.shape}")
    # print(f"Correct prediction: {item['correct_prediction']}, Predicted label: {item['pred_label']}\n")
    # print("*" * 80)

    # Clear references to large attention tensors
    del full_inputs, outputs, h_layer, attn_layer, premise_contexts, step_embeds, clause_embeds, M, M_col_norm

with open(output_pkl, "wb") as outfile:
    pickle.dump(results, outfile)

print(f"Features saved successfully to {output_pkl}")

100%|██████████| 400/400 [02:45<00:00,  2.41it/s]


Features saved successfully to ./results/400_samples_SATBench_Llama3.1-8B/dual_stream_features_layer31.pkl


In [1]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class DualStreamRelationalProbe(nn.Module):
    def __init__(self, d_model, d_cov=12, d_proj=256, gru_hidden=128, dropout=0.2):
        super().__init__()
        
        # --- Stream 1: Step Transition Verifier ---
        # Projects z_t = [h_{t-1}, h_t, h_t - h_{t-1}, p_t, h_t * p_t]
        self.step_projector = nn.Sequential(
            nn.Linear(5 * d_model, d_proj),
            nn.LayerNorm(d_proj),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.gru = nn.GRU(
            input_size=d_proj,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        # Gate to compute Attention Pooling weights over the steps
        self.step_gate = nn.Linear(2 * gru_hidden, 1)

        # --- Stream 2: Global Coverage Branch ---
        # Processes the 12-dim attention spectral features
        self.cov_mlp = nn.Sequential(
            nn.Linear(d_cov, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, 64),
            nn.GELU()
        )

        # --- Fusion Head ---
        # Attn Pool (2*gru_hidden) + Min Pool (2*gru_hidden) + Coverage (64)
        fusion_dim = (4 * gru_hidden) + 64
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)  # Output size 2 for CrossEntropy
        )

    def forward(self, step_embeds, premise_contexts, lengths, cov_features):
        B, T_max, D = step_embeds.shape

        # Shift sequence by 1 to align h_{t-1} with h_t
        prev_step_embeds = torch.zeros_like(step_embeds)
        prev_step_embeds[:, 1:, :] = step_embeds[:, :-1, :]

        delta_h = step_embeds - prev_step_embeds
        hadamard_hp = step_embeds * premise_contexts

        # Relational tuple z_t
        z = torch.cat([
            prev_step_embeds, 
            step_embeds, 
            delta_h, 
            premise_contexts, 
            hadamard_hp
        ], dim=-1)
        
        z_proj = self.step_projector(z)

        # Process valid sequence lengths through BiGRU
        packed = pack_padded_sequence(z_proj, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.gru(packed)
        gru_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T_max)

        mask = torch.arange(T_max, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)

        # 1. Attention Pooling
        gate_logits = self.step_gate(gru_out).squeeze(-1)
        gate_logits = gate_logits.masked_fill(~mask, float("-inf"))
        attn_weights = torch.softmax(gate_logits, dim=-1)
        attn_pooled = torch.bmm(attn_weights.unsqueeze(1), gru_out).squeeze(1)

        # 2. Min Pooling (Catches the single weakest logical link)
        masked_gru_for_min = gru_out.masked_fill(~mask.unsqueeze(-1), float("inf"))
        min_pooled = masked_gru_for_min.min(dim=1).values

        # 3. Global Coverage Features
        cov_repr = self.cov_mlp(cov_features)

        # Final Fusion
        fused = torch.cat([attn_pooled, min_pooled, cov_repr], dim=-1)
        logits = self.classifier(fused)

        return logits

In [10]:
from model import *
import os
import torch
import numpy as np
from data_reader import load_hidden_states
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

gpu = "1"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

batch_size = 128
learning_rate = 1e-4
weight_decay = 1e-2

def gen_classifier_roc(dataset_dict, labels, model_class, save_name="dual_stream_model.pt"):
    indices = np.arange(len(labels))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=123)
    
    # Initialize Model
    d_model = dataset_dict['step_embeds'].shape[-1]
    classifier_model = model_class(d_model=d_model).to(device)
    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Train Split
    X_steps_train = dataset_dict['step_embeds'][train_idx].to(device)
    X_premise_train = dataset_dict['premise_contexts'][train_idx].to(device)
    lengths_train = dataset_dict['lengths'][train_idx].to(device)
    cov_train = dataset_dict['cov_features'][train_idx].to(device)
    y_train = torch.tensor(labels[train_idx], dtype=torch.long, device=device)

    # Test Split
    X_steps_test = dataset_dict['step_embeds'][test_idx].to(device)
    X_premise_test = dataset_dict['premise_contexts'][test_idx].to(device)
    lengths_test = dataset_dict['lengths'][test_idx].to(device)
    cov_test = dataset_dict['cov_features'][test_idx].to(device)
    y_test = torch.tensor(labels[test_idx], dtype=torch.long, device=device)

    classifier_model.train()
    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample_idx = torch.randperm(X_steps_train.shape[0])[:batch_size]
        
        pred = classifier_model(
            X_steps_train[sample_idx],
            X_premise_train[sample_idx],
            lengths_train[sample_idx],
            cov_train[sample_idx]
        )
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample_idx])
        loss.backward()
        optimizer.step()

    # torch.save(classifier_model.state_dict(), f"{save_model_dir}/{save_name}")

    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(
            X_steps_test, X_premise_test, lengths_test, cov_test
        ), dim=1)
        
        prediction_classes = (pred[:, 1] > 0.5).type(torch.long).cpu()
        roc = roc_auc_score(y_test.cpu(), pred[:, 1].cpu())
        acc = (prediction_classes.numpy() == y_test.cpu().numpy()).mean()
        
        return roc, acc, prediction_classes.numpy(), y_test.cpu().numpy()

Using device: cuda:1


In [12]:
results_file = "./results/400_samples_SATBench_Llama3.1-8B/dual_stream_features_layer31.pkl"
all_results = {}
try:
    classifier_results = {}
    print(f"Loading data from {results_file}")
    
    # load_hidden_states must be updated to return all keys from the new extractor
    results, correct_list = load_hidden_states(results_file)
    
    seq_steps = []
    seq_premises = []
    cov_feats = []
    lengths_list = []
    d_model = 4096 # Llama-3.1-8B dim
    
    for i in range(len(results['generated_embeddings'])):
        steps = results['generated_embeddings'][i]
        
        # NOTE: This assumes your new extractor saves 'premise_contexts' and 'cov_features'
        # If running on the old PKL, you MUST re-extract first.
        premises = results.get('premise_contexts', [np.zeros_like(steps)])[i] 
        cov = results.get('cov_features', [np.zeros(12, dtype=np.float32)])[i]
        
        # Handle empty reasoning traces
        if steps.shape[0] == 0:
            steps = np.zeros((1, d_model), dtype=np.float32)
            premises = np.zeros((1, d_model), dtype=np.float32)
            
        seq_steps.append(torch.tensor(steps, dtype=torch.float32))
        seq_premises.append(torch.tensor(premises, dtype=torch.float32))
        cov_feats.append(torch.tensor(cov, dtype=torch.float32))
        lengths_list.append(steps.shape[0])

    # Pad sequences to max length in the batch -> Shape: [B, T_max, hidden_size]
    dataset_dict = {
        'step_embeds': torch.nn.utils.rnn.pad_sequence(seq_steps, batch_first=True),
        'premise_contexts': torch.nn.utils.rnn.pad_sequence(seq_premises, batch_first=True),
        'lengths': torch.tensor(lengths_list, dtype=torch.long),
        'cov_features': torch.stack(cov_feats, dim=0)
    }
    
    correct_array = np.array(correct_list)

    # Train Dual Stream Classifier
    context_emb_roc, context_emb_acc, prediction_classes, y_true = gen_classifier_roc(
        dataset_dict=dataset_dict, 
        labels=correct_array, 
        model_class=DualStreamRelationalProbe, 
        save_name="dual_stream_context_emb.pt"
    )
    
    confusion_stats, confusion_matrix_values = compute_confusion_stats(y_true, prediction_classes, labels=(0, 1))
    plot_confusion_matrix(
        confusion_matrix_values,
        labels=("Hallucinated", "Non-hallucinated"),
        save_path="./results/400_samples_SATBench_Llama3.1-8B/confusion_matrix_hallu.png"
    )

    classifier_results['context_emb_roc'] = context_emb_roc
    classifier_results['context_emb_acc'] = context_emb_acc
    classifier_results.update({k: confusion_stats[k] for k in ['TP', 'TN', 'FP', 'FN']})
    
    all_results[results_file] = classifier_results.copy()
except Exception as err:
    print(err)

Loading data from ./results/400_samples_SATBench_Llama3.1-8B/dual_stream_features_layer31.pkl


100%|██████████| 1001/1001 [01:20<00:00, 12.45it/s]

name 'compute_confusion_stats' is not defined


In [13]:
print(f"AUC ROC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")

AUC ROC: 0.5374, Accuracy: 0.4627


### Confusion Matrix

In [9]:
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
import os
import numpy as np

data = json.load(open("./results/2100_samples_SATBench_Llama3.1-8B/logical_result_max512_temp0.0_removed_none.json", "r"))

pred_labels = []
true_labels = []

for idx in tqdm(test_indices):
    item = data[idx]
    if item["pred_label"]:
        pred_label = item["pred_label"]
        if item["correct_prediction"]:
            if pred_label == "SAT":
                pred_labels.append(1)
                true_labels.append(1)
            else:
                pred_labels.append(0)
                true_labels.append(0)
        else:
            if pred_label == "SAT":
                pred_labels.append(1)
                true_labels.append(0)
            else:
                pred_labels.append(0)
                true_labels.append(1)

100%|██████████| 333/333 [00:00<00:00, 412006.85it/s]


In [11]:
len(true_labels), len(pred_labels)

(333, 333)

In [12]:
cm = confusion_matrix(true_labels, pred_labels)
tn, fp, fn, tp = cm.ravel()
print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

label = "LLM output"
confusion_plot_path = plot_confusion_matrix(
    cm,
    labels=("Unsatisfiable", "Satisfiable"),
    save_path=f"./results/2100_samples_SATBench_Llama3.1-8B/max_signal/confusion_matrix_{label}.png",
    label=label
)

TP: 114, TN: 65, FP: 111, FN: 43


### Classifier as Reasoner

In [ ]:
from model import *
import os
import torch
import numpy as np
from data_reader import load_hidden_states
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

classifier = SingleMLP_Classifier
results_file = "./results/2100_samples_SATBench_Llama3.1-8B/contextual_embeddings_max1024_temp0.0.pkl"
classifier_model = classifier(4096).to(device)
checkpoint = "./models/Llama-3.1-8B-Instruct/mlp_context_emb.pt"
state_dict = torch.load(checkpoint, map_location=device)
classifier_model.load_state_dict(state_dict)

In [ ]:
results, correct = load_hidden_states(results_file)
pooled_embeddings = []
for i in range(len(results['generated_embeddings'])):
    embeddings = results['generated_embeddings'][i]
    emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
    pooled_emb = torch.mean(emb_sequence, dim=0)
    pooled_embeddings.append(pooled_emb)
pooled_embeddings_tensor = torch.stack(pooled_embeddings)

labels = correct.astype(int)
inputs = torch.tensor(pooled_embeddings_tensor).to(device)
labels = torch.tensor(labels).to(device)

classifier_model.eval()
with torch.no_grad():
    pred = torch.nn.functional.softmax(classifier_model(inputs), dim=1)
    prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()

context_emb_roc, context_emb_acc = roc_auc_score(labels.cpu(), pred[:,1].cpu()), (prediction_classes.numpy()==labels.cpu().numpy()).mean()
print(f"Contextual Embeddings ROC AUC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Define the model you want to use
model_name = "meta-llama/Llama-3.1-8B-Instruct"  # You can substitute this with any chat/instruct model

# 2. Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto", 
    device_map="auto",
    cache_dir="./.cache/models",
)

In [ ]:
scenario = "In a botanical research facility, there are three rare plant species being studied: Orchid, Fern, and Cactus. Researchers are independently deciding whether each plant species exhibits two specific growth patterns: diurnal growth (0) or nocturnal growth (1). Each plant species can show either, both, or neither pattern without any constraints or implications on the others."
conditions = [
"1. Cactus exhibits either diurnal growth or nocturnal growth.",
"2. Cactus does not exhibit diurnal growth.",
"3. Either Cactus does not exhibit diurnal growth, or Cactus does not exhibit nocturnal growth.",
"4. Cactus does not exhibit nocturnal growth."
]
question = "Is there a way to assign growth patterns to the Cactus that satisfies all these conditions?"

if "Llama" in model_name:
    system_prompt = f"""You are a logical reasoning assistant. You are given a logic puzzle.
    Guidelines:
    - All constraints come **only** from the <conditions> section.
    - The <scenario> provides background and intuition, but **does not impose any additional rules or constraints**.
    - All variables represent **independent decisions**; there is no mutual exclusivity or implicit linkage unless stated explicitly in <conditions>.
    - Variables not mentioned in <conditions> are considered unknown and irrelevant to satisfiability.
    Your task:
    - If the puzzle is satisfiable, propose one valid assignment that satisfies all the conditions.
    - If the puzzle is unsatisfiable, explain why some of the conditions cannot all be true at once.
    Think step by step but don't return any intermediate reasoning, only give a brief response. At the end of your response, output exactly one of the following labels on a new line:
    [SAT]- if a valid assignment exists
    [UNSAT]- if the constraints cannot be satisfied
    Do not add any text or formatting after the final label.
    """
else:
    system_prompt = f"""You are a logical reasoning assistant. You are given a logic puzzle.
Guidelines:
- All constraints come **only** from the <conditions> section.
- The <scenario> provides background and intuition, but **does not impose any additional rules or constraints**.
- All variables represent **independent decisions**; there is no mutual exclusivity or implicit linkage unless stated explicitly in <conditions>.
- Variables not mentioned in <conditions> are considered unknown and irrelevant to satisfiability.
Your task:
- If the puzzle is satisfiable, propose one valid assignment that satisfies all the conditions.
- If the puzzle is unsatisfiable, explain why some of the conditions cannot all be true at once.
Think step by step but don't return any intermediate reasoning, summarize the key points, and only give a brief response. At the end of your response, output exactly one of the following labels on a new line:
[SAT] - if a valid assignment exists
[UNSAT] - if the constraints cannot be satisfied
Do not add any text or formatting after the final label.
"""
    
user_prompt = f"""<scenario>
{scenario}
<conditions>
{conditions}
<question>
{question}
"""

In [ ]:
# 3. Structure your conversation with 'system' and 'user' roles
messages = [
    {
        "role": "system", 
        "content": system_prompt
    },
    {
        "role": "user", 
        "content": user_prompt
    }
]

# 4. Apply the chat template to automatically inject the correct model tags
tokenized_chat = tokenizer.apply_chat_template(
    messages, 
    tokenize=True, 
    add_generation_prompt=True,  # Tells the model it is its turn to reply
    return_tensors="pt"
).to(model.device)

# 5. Generate the response
outputs = model.generate(
    **tokenized_chat,
    max_new_tokens=500,
    do_sample=False,
    temperature=0.0,
    pad_token_id=tokenizer.eos_token_id
)

# 6. Decode and print the result
# We slice outputs to only decode tokens generated *after* your prompt
# prompt_length = tokenized_chat.shape[1]
prompt_length = len(tokenized_chat.input_ids[0])
response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

print(response)

From condition 2 and 4, we can conclude that Cactus does not exhibit diurnal growth and does not exhibit nocturnal growth. This directly contradicts condition 1, which states that Cactus exhibits either diurnal growth or nocturnal growth.

[UNSAT]


In [26]:
tokenized_chat.input_ids.shape[-1]

405